# FedCure: coalitions with participation-aware scheduling (M=3)

Clients are grouped under edge servers into **coalitions with similar label mixes**. Each coalition trains synchronously for a few edge rounds, the cloud mixes the coalitions' models in asynchronously, and a scheduler decides which coalition trains next so that every coalition takes part fairly. Based on Chen et al., *FedCure: Mitigating Participation Bias in Semi-Asynchronous Federated Learning with Non-IID Data* (arXiv 2511.10227, 2025).

**How it works**

1. **Coalition formation (before training).** Clients start grouped by their most common label (one edge server per kind of data). Then clients switch edge servers whenever that lowers the average Jensen-Shannon divergence between the coalitions' label distributions, until no switch helps.
2. **Edge rounds.** A scheduled coalition receives the global model; its clients train, the edge server averages them (FedAvg), and this repeats `group_rounds` times.
3. **Cloud update.** The edge model is mixed into the global model: `global = (1 - w) * global + w * edge`, with `w = mix * decay ** staleness`.
4. **Scheduling.** Each coalition has a virtual queue that grows by its target share `kappa * |D_m| / |D|` every round and shrinks by 1 when it is scheduled. The next coalition is the idle one with the largest `queue + beta * (1 - estimated_latency / max_latency)`: long-neglected coalitions first, fast ones preferred when the queues are similar.

> **Note:** coalitions with similar label mixes make every edge model a good summary of all the data, and the queues stop fast coalitions from taking every turn. Together they target **participation bias**: the global model drifting toward the data of fast clients.

> **Caution:** simplifications compared with the paper:
> * 3 edge servers and 2 edge rounds instead of 5 and 12: with 10 clients and 100 client updates, 12 edge rounds would leave only about 3 cloud updates.
> * `active_coalitions = 2`: two coalitions train at once, so the scheduler always chooses between the idle ones. The paper schedules one coalition per round from the available ones but does not say how many train at once.
> * Device speeds do not change here, so the paper's Bayesian latency estimate is a running mean of each coalition's observed job time (the posterior mean under a weak prior). `max_latency` stands in for the paper's normaliser `I`, which is not defined further.
> * `kappa` is not given in the paper. With 0.9, the largest coalition was promised 49% of the turns but could take only about 40%, because it is still training during most decisions; its queue grew without bound, so the long-run balance was not met. With 0.6 every coalition reaches its share and all queues stay small. If a coalition's queue keeps growing, its target cannot be met: lower `kappa`.
> * The CPU-frequency allocation (energy saving) is not simulated: devices always run at full speed.
> * The cloud mixing weight is small (`mix = 0.2`), and only 2 of 3 coalitions train at once, so FedCure needs more virtual time for the same number of client updates than the other asynchronous approaches.

## 1. Load the simulator

Gives this notebook everything from `hierarchical_simulator.ipynb`: all of `simulator.ipynb` (settings, `run`, ...) plus `GroupApproach` for group-based approaches.

In [1]:
%run ../hierarchical_simulator.ipynb

## 2. The approach

Parameters are the class attributes at the top. Change them here, or pass them when running (see the tip below).

In [2]:
class FedCure(GroupApproach):
    name = "fedcure"
    label = "FedCure coalitions M=3"          # shown in plot legends
    edge_servers = 3              # number of coalitions (M in the paper)
    group_rounds = 2              # edge rounds before a coalition reports to the cloud (12 in the paper)
    active_coalitions = 2         # coalitions training at once; the scheduler picks among the idle ones
    mix = 0.2                     # initial cloud mixing weight (l in the paper)
    decay = 0.9                   # staleness penalty: weight = mix * decay ** staleness (k in the paper)
    beta = 0.5                    # weight of efficiency against balance in scheduling
    kappa = 0.6                   # share of turns guaranteed for balance: coalition m gets at least kappa * |D_m| / |D|
                                  # (0.9 promised the largest coalition more turns than it can take while still training)
    max_switches = 1000           # most client switches during coalition formation (L in the paper)

    # ---------- 1. coalition formation: minimise the label-distribution divergence between coalitions ----------
    def form_groups(self, server):
        counts = server.data.label_mix * np.array([c.samples for c in server.clients])[:, None]
        # start "edge non-IID" (as in the paper): clients grouped by their most common label
        order = sorted(range(len(server.clients)), key=lambda i: (counts[i].argmax(), i))
        edges = np.linspace(0, len(order), self.edge_servers + 1).round().astype(int)
        groups = [order[a:b] for a, b in zip(edges[:-1], edges[1:])]
        self.js_start = js = self.divergence(groups, counts)
        switches = 0
        while switches < self.max_switches:                # switch clients while that lowers the divergence
            improved = False
            for n in server.rng.permutation(len(server.clients)):
                here = next(g for g, members in enumerate(groups) if n in members)
                if len(groups[here]) == 1:
                    continue                               # every edge server keeps at least one client
                best, best_js = here, js
                for there in range(len(groups)):
                    if there != here:
                        trial = [list(m) for m in groups]
                        trial[here].remove(n)
                        trial[there].append(n)
                        trial_js = self.divergence(trial, counts)
                        if trial_js < best_js - 1e-12:
                            best, best_js = there, trial_js
                if best != here:
                    groups[here].remove(n)
                    groups[best].append(n)
                    js, improved, switches = best_js, True, switches + 1
                    if switches == self.max_switches:
                        break
            if not improved:
                break                                      # a full pass without a switch: the partition is stable
        self.js_final = js
        return [sorted(int(c) for c in g) for g in groups]

    @staticmethod
    def divergence(groups, counts):
        """Average Jensen-Shannon divergence between the coalitions' label distributions (Eq. 3)."""
        q = [counts[g].sum(0) / counts[g].sum() for g in groups]
        def kl(a, b):
            mask = a > 0
            return float((a[mask] * np.log(a[mask] / b[mask])).sum())
        pairs = [(i, j) for i in range(len(q)) for j in range(i + 1, len(q))]
        return sum(0.5 * kl(q[i], (q[i] + q[j]) / 2) + 0.5 * kl(q[j], (q[i] + q[j]) / 2) for i, j in pairs) / len(pairs)

    # ---------- 2. cloud update: staleness-weighted mixing of the edge model ----------
    def on_group_done(self, server, g, model, updates):
        weight = self.mix * self.decay ** self.group_staleness(server, g)
        server.set_model(average_models([server.model, model], [1 - weight, weight]), updates)
        self.latency[g].append(server.time - self.job_start[g])

    # ---------- 3. scheduling: virtual queues (balance) + estimated latency (efficiency) ----------
    def start(self, server):
        self.latency, self.job_start = {}, {}              # observed job durations, start time of the current job
        super().start(server)

    def start_group(self, server, g):
        self.job_start[g] = server.time
        self.latency.setdefault(g, [])
        super().start_group(server, g)

    def first_groups(self, server):
        total = sum(c.samples for c in server.clients)
        self.target = [self.kappa * sum(server.clients[c].samples for c in members) / total for members in self.groups]
        self.queue = [0.0] * len(self.groups)              # max(-delta, 0) = 0
        return range(len(self.groups))                     # the paper schedules every coalition in round 0

    def estimated_latency(self, g):
        seen = [np.mean(v) for v in self.latency.values() if v]
        if self.latency[g]:
            return float(np.mean(self.latency[g]))         # posterior mean with a weak prior
        return float(np.mean(seen)) if seen else 0.0       # not observed yet: assume the average

    def next_groups(self, server, g):
        chosen = []
        while len(self.active) + len(chosen) < self.active_coalitions:
            idle = [m for m in range(len(self.groups)) if m not in self.active and m not in chosen]
            if not idle:
                break
            est = {m: self.estimated_latency(m) for m in idle}
            norm = max(est.values()) or 1.0
            pick = max(idle, key=lambda m: self.queue[m] + self.beta * (1 - est[m] / norm))
            for m in range(len(self.groups)):              # Eq. 13: the queue grows when a coalition is skipped
                self.queue[m] = max(self.queue[m] + self.target[m] - (m == pick), 0.0)
            chosen.append(pick)
        return chosen

## 3. Run it

Runs every seed listed in `settings.yaml` and saves the results to `results/`. Then open **`compare.ipynb`** to see it next to the other approaches.

> **Tip:** to try a variant without editing the class, run e.g. `run(FedCure(active_coalitions=3))` to let all coalitions train at once (faster, but no scheduling choice), or `run(FedCure(group_rounds=12))` for the paper's edge rounds. It is saved as a separate run.

In [3]:
results = run(FedCure())

fedcure                      seed=42   val= 92.20%  acc= 93.15%  time=  1539.8s  updates=  100  worst10%= 92.39%  wall=18.2s


## 4. Quick look

The final evaluation for each seed:

In [4]:
results.groupby("seed").tail(1)[["seed", "time", "updates", "accuracy", "loss", "worst10_client_acc", "mean_staleness"]].round(4)

,seed,time,updates,accuracy,loss,worst10_client_acc,mean_staleness
13,42,1539.7608,100,0.9315,0.2172,0.9239,0.1
